# 4.3 Feature Scaling, Encoding & Engineering

## Table of Contents
- [4.3.1 Feature Scaling Rationale & Methods](#431-feature-scaling-rationale--methods)
- [4.3.2 Categorical Encoding Overview](#432-categorical-encoding-overview)
- [4.3.3 Feature Engineering Basics](#433-feature-engineering-basics)
- [Knowledge Check](#knowledge-check)
- [Mini-Challenges](#mini-challenges)
- [Practical Connections](#practical-connections)

## Introduction

After handling missing values, the next critical steps in data preparation are feature scaling, categorical encoding, and feature engineering. These transformations help ensure that your data is in the optimal format for machine learning algorithms, potentially improving model performance significantly.

Let's import the necessary libraries:

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler, 
    MinMaxScaler, 
    RobustScaler,
    OneHotEncoder, 
    LabelEncoder,
    PolynomialFeatures
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 4.3.1 Feature Scaling Rationale & Methods

### What is Feature Scaling?
Feature scaling is the process of transforming numerical features to be on a similar scale or range.

### Why is Feature Scaling Needed?

#### 1. Algorithms Sensitive to Scale
Algorithms that use distance calculations or rely on gradient descent optimization can be heavily influenced by features with larger ranges:

* **Distance-based algorithms:**
  - K-Nearest Neighbors (KNN)
  - K-Means Clustering
  - Support Vector Machines (SVM)
  
* **Gradient descent-based algorithms:**
  - Linear Regression
  - Logistic Regression
  - Neural Networks

#### 2. The Problem with Unscaled Features

Features with larger values might:
- Dominate the distance calculation
- Cause slower convergence during optimization
- Lead to unstable parameter estimates
- Make regularization ineffective

#### 3. Which Algorithms Require Scaling?

| **Need Scaling** | **Scale Invariant** |
|------------------|---------------------|
| Linear Regression (with gradient descent) | Decision Trees |
| Logistic Regression | Random Forests |
| Neural Networks | Gradient Boosting |
| K-Nearest Neighbors | Naive Bayes |
| K-Means Clustering | |
| Principal Component Analysis | |
| Support Vector Machines | |

### Common Scaling Methods

#### 1. Standardization (Z-score Scaling)
- **Formula:** $X_{scaled} = \frac{X - \mu}{\sigma}$ (where μ is the mean and σ is the standard deviation)
- **Result:** Transforms data to have a mean of 0 and a standard deviation of 1
- **Implementation:** `sklearn.preprocessing.StandardScaler`
- **Pros:** Centers data around zero; commonly used
- **Cons:** Not robust to outliers (mean/std dev are sensitive); doesn't guarantee a fixed range

#### 2. Normalization (Min-Max Scaling)
- **Formula:** $X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}$
- **Result:** Rescales data to a fixed range, typically [0, 1]
- **Implementation:** `sklearn.preprocessing.MinMaxScaler`
- **Pros:** Guarantees data is within a specific range; useful for algorithms expecting inputs in [0, 1]
- **Cons:** Highly sensitive to outliers (min/max values dictate the range); doesn't center data around zero

#### 3. Robust Scaling
- **Formula:** $X_{scaled} = \frac{X - median(X)}{IQR(X)}$ (where IQR is the interquartile range)
- **Result:** Similar to standardization but uses median and IQR instead of mean and std dev
- **Implementation:** `sklearn.preprocessing.RobustScaler`
- **Pros:** Less influenced by outliers than standardization or min-max scaling
- **Cons:** May not be as efficient for normally distributed data

Let's visualize and compare these scaling methods:

In [ ]:
# Load the California Housing dataset
data = fetch_california_housing()
X = data.data
y = data.target
feature_names = data.feature_names

# Select a few features for demonstration
selected_features = ['MedInc', 'AveRooms', 'AveOccup', 'Latitude']
selected_indices = [feature_names.index(feat) for feat in selected_features]
X_selected = X[:, selected_indices]

# Create a DataFrame
df = pd.DataFrame(X_selected, columns=selected_features)

# Display basic statistics
print("Original Data Statistics:")
print(df.describe().round(2))

# Create scatter plots with original data
plt.figure(figsize=(12, 10))
plt.subplot(3, 2, 1)
plt.scatter(df['MedInc'], df['AveRooms'], alpha=0.5, s=10)
plt.title('Original Data: MedInc vs AveRooms')
plt.xlabel('MedInc (Median Income)')
plt.ylabel('AveRooms (Average Rooms)')

plt.subplot(3, 2, 2)
plt.scatter(df['Latitude'], df['AveOccup'], alpha=0.5, s=10)
plt.title('Original Data: Latitude vs AveOccup')
plt.xlabel('Latitude')
plt.ylabel('AveOccup (Average Occupancy)')

# Apply StandardScaler
scaler1 = StandardScaler()
df_std = pd.DataFrame(
    scaler1.fit_transform(df),
    columns=df.columns
)

plt.subplot(3, 2, 3)
plt.scatter(df_std['MedInc'], df_std['AveRooms'], alpha=0.5, s=10)
plt.title('Standardized: MedInc vs AveRooms')
plt.xlabel('MedInc (Standardized)')
plt.ylabel('AveRooms (Standardized)')

plt.subplot(3, 2, 4)
plt.scatter(df_std['Latitude'], df_std['AveOccup'], alpha=0.5, s=10)
plt.title('Standardized: Latitude vs AveOccup')
plt.xlabel('Latitude (Standardized)')
plt.ylabel('AveOccup (Standardized)')

# Apply MinMaxScaler
scaler2 = MinMaxScaler()
df_minmax = pd.DataFrame(
    scaler2.fit_transform(df),
    columns=df.columns
)

plt.subplot(3, 2, 5)
plt.scatter(df_minmax['MedInc'], df_minmax['AveRooms'], alpha=0.5, s=10)
plt.title('Min-Max Scaled: MedInc vs AveRooms')
plt.xlabel('MedInc (Min-Max Scaled)')
plt.ylabel('AveRooms (Min-Max Scaled)')

plt.subplot(3, 2, 6)
plt.scatter(df_minmax['Latitude'], df_minmax['AveOccup'], alpha=0.5, s=10)
plt.title('Min-Max Scaled: Latitude vs AveOccup')
plt.xlabel('Latitude (Min-Max Scaled)')
plt.ylabel('AveOccup (Min-Max Scaled)')

plt.tight_layout()
plt.show()

### Impact of Scaling on Model Performance

Let's demonstrate how scaling affects the performance of different algorithms:

In [ ]:
# Load the Breast Cancer dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Function to evaluate models with different scaling
def evaluate_with_scaling(models, scalers):
    results = []
    
    for model_name, model in models.items():
        for scaler_name, scaler in scalers.items():
            # Scale the data if a scaler is provided
            if scaler is not None:
                X_train_scaled = scaler.fit_transform(X_train)
                X_test_scaled = scaler.transform(X_test)
            else:
                X_train_scaled = X_train
                X_test_scaled = X_test
            
            # Train and evaluate the model
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            accuracy = accuracy_score(y_test, y_pred)
            
            # Store the results
            results.append({
                'Model': model_name,
                'Scaler': scaler_name,
                'Accuracy': accuracy
            })
    
    # Convert to DataFrame for easier display
    return pd.DataFrame(results)

# Define models and scalers
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'SVM (linear)': SVC(kernel='linear', random_state=42)
}

scalers = {
    'No Scaling': None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

# Evaluate models with different scaling
results = evaluate_with_scaling(models, scalers)

# Display results
print("Impact of Scaling on Model Performance:")
print(results.round(4))

# Visualize results
plt.figure(figsize=(12, 6))
pivot_results = results.pivot(index='Model', columns='Scaler', values='Accuracy')
sns.heatmap(pivot_results, annot=True, cmap='YlGnBu', fmt='.4f')
plt.title('Model Accuracy with Different Scaling Methods')
plt.tight_layout()
plt.show()

# Bar plot comparison
plt.figure(figsize=(12, 6))
sns.barplot(x='Model', y='Accuracy', hue='Scaler', data=results)
plt.title('Impact of Scaling on Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Model')
plt.xticks(rotation=45)
plt.legend(title='Scaling Method')
plt.tight_layout()
plt.show()

### Key Best Practices for Feature Scaling

1. **Fit scaler only on training data**: Always fit scalers on the training set and then apply the same transformation to the test set to prevent data leakage.

2. **Choose the right scaler for your data**:
   - Use **StandardScaler** for most cases, especially when features follow normal distribution
   - Use **MinMaxScaler** when you need a specific bounded range [0,1]
   - Use **RobustScaler** when your data has many outliers

3. **Scale before dimensionality reduction**: Always scale before applying PCA or other dimensionality reduction techniques.

4. **Scale before regularization**: For regularized models (Ridge, Lasso), scaling is essential for regularization to work properly.

5. **Some algorithms don't require scaling**:
   - Tree-based methods (Decision Trees, Random Forests, Gradient Boosting)
   - Naive Bayes

## 4.3.2 Categorical Encoding Overview

### What is Categorical Encoding?
Categorical encoding is the process of converting categorical features (text labels) into numerical representations that machine learning algorithms can understand.

### Why is Categorical Encoding Needed?
Most machine learning algorithms work only with numerical data and cannot directly process categorical features.

### Common Encoding Methods

#### 1. One-Hot Encoding (OHE)
- **Concept**: Creates a new binary (0 or 1) column for *each unique category* in the original feature. For a given row, the column corresponding to its category gets a 1, and all other new columns get a 0.
- **Implementation**: `sklearn.preprocessing.OneHotEncoder` or `pandas.get_dummies()`
- **Pros**: Doesn't assume any ordering between categories; works well with most algorithms
- **Cons**: Can create a very large number of new features if the original feature has many unique categories (high cardinality)

#### 2. Label Encoding
- **Concept**: Assigns a unique integer to each category (e.g., Red=0, Green=1, Blue=2)
- **Implementation**: `sklearn.preprocessing.LabelEncoder`
- **Pros**: Simple; doesn't increase dimensionality
- **Cons**: Introduces an *arbitrary ordinal relationship* (implies Blue > Green > Red); generally suitable only for ordinal categorical features

#### 3. Binary Encoding
- **Concept**: Represents each category as a binary code (e.g., A=00, B=01, C=10, D=11)
- **Pro**: More compact than one-hot encoding for high-cardinality features
- **Con**: Loses the direct interpretability of one-hot encoding

#### 4. Target Encoding
- **Concept**: Replaces a category with the mean target value for that category
- **Pros**: Works well for high-cardinality features; can capture information in the ordering of labels
- **Cons**: Risk of overfitting; requires careful cross-validation

Let's demonstrate these encoding methods:

In [ ]:
# Create a sample dataset with categorical features
np.random.seed(42)
n_samples = 500

# Create a DataFrame with a mix of numerical and categorical features
data = pd.DataFrame({
    'age': np.random.normal(40, 10, n_samples),
    'income': np.random.normal(60000, 15000, n_samples),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples),
    'job_sector': np.random.choice(['Tech', 'Finance', 'Healthcare', 'Education', 'Other'], n_samples),
    'marital_status': np.random.choice(['Single', 'Married', 'Divorced', 'Widowed'], n_samples)
})

# Add a binary target variable (loan approval)
data['loan_approved'] = data.apply(generate_target, axis=1)

# Display the first few rows
print("Sample data:")
print(data.head())

# Check data types and categorical feature counts
print("\nData types:")
print(data.dtypes)

print("\nCategorical feature value counts:")
for col in ['education', 'job_sector', 'marital_status']:
    print(f"\n{col} value counts:")
    print(data[col].value_counts())

# 1. One-Hot Encoding using pandas get_dummies
data_onehot = pd.get_dummies(data, columns=['education', 'job_sector', 'marital_status'], drop_first=False)

# Display the first few rows after one-hot encoding
print("\nAfter One-Hot Encoding (first 5 columns):")
print(data_onehot.iloc[:5, :10])
print(f"Shape after one-hot encoding: {data_onehot.shape}")

# 2. Label Encoding
from sklearn.preprocessing import LabelEncoder

data_label = data.copy()
label_encoders = {}

for col in ['education', 'job_sector', 'marital_status']:
    label_encoders[col] = LabelEncoder()
    data_label[col] = label_encoders[col].fit_transform(data_label[col])

# Display the first few rows after label encoding
print("\nAfter Label Encoding:")
print(data_label.head())

# Show mapping for education
print("\nLabel Encoder Mapping for 'education':")
for i, label in enumerate(label_encoders['education'].classes_):
    print(f"{label} -> {i}")

# 3. Target Encoding (mean encoding)
def target_encode(train_df, test_df, cols, target_col, min_samples=10, smoothing=10):
    # Dictionary to store the encoders
    encoders = {}
    
    # Process each column
    for col in cols:
        # Calculate the global mean
        global_mean = train_df[target_col].mean()
        
        # Group by the column and calculate the mean of the target
        category_means = train_df.groupby(col)[target_col].agg(['mean', 'count'])
        
        # Apply smoothing
        smoothed_means = (category_means['mean'] * category_means['count'] + 
                         global_mean * smoothing) / (category_means['count'] + smoothing)
        
        # Create the encoder dictionary
        encoders[col] = smoothed_means.to_dict()
        
        # Apply encoding to train data
        train_df[f"{col}_target_encoded"] = train_df[col].map(encoders[col]).fillna(global_mean)
        
        # Apply encoding to test data
        test_df[f"{col}_target_encoded"] = test_df[col].map(encoders[col]).fillna(global_mean)
    
    return train_df, test_df, encoders

# Split the data for target encoding demonstration
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Apply target encoding
train_encoded, test_encoded, encoders = target_encode(
    train_data.copy(), 
    test_data.copy(), 
    cols=['education', 'job_sector', 'marital_status'],
    target_col='loan_approved'
)

# Display the first few rows after target encoding
print("\nAfter Target Encoding (Training Data):")
print(train_encoded.head())

# Compare the different encoding methods
plt.figure(figsize=(15, 5))

# Original distribution (for education)
plt.subplot(1, 3, 1)
sns.countplot(x='education', data=data)
plt.title('Original Categories')
plt.xticks(rotation=45)

# Label Encoded
plt.subplot(1, 3, 2)
sns.countplot(x='education', data=data_label)
plt.title('Label Encoded')

# Target Encoded
plt.subplot(1, 3, 3)
sns.boxplot(x='education', y='education_target_encoded', data=train_encoded)
plt.title('Target Encoded')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Visualize the relationship between categorical features and target
plt.figure(figsize=(15, 5))

# Education vs. Target
plt.subplot(1, 3, 1)
target_by_edu = data.groupby('education')['loan_approved'].mean().reset_index()
sns.barplot(x='education', y='loan_approved', data=target_by_edu)
plt.title('Loan Approval Rate by Education')
plt.ylabel('Approval Rate')
plt.xticks(rotation=45)

# Job Sector vs. Target
plt.subplot(1, 3, 2)
target_by_job = data.groupby('job_sector')['loan_approved'].mean().reset_index()
sns.barplot(x='job_sector', y='loan_approved', data=target_by_job)
plt.title('Loan Approval Rate by Job Sector')
plt.ylabel('Approval Rate')
plt.xticks(rotation=45)

# Marital Status vs. Target
plt.subplot(1, 3, 3)
target_by_marital = data.groupby('marital_status')['loan_approved'].mean().reset_index()
sns.barplot(x='marital_status', y='loan_approved', data=target_by_marital)
plt.title('Loan Approval Rate by Marital Status')
plt.ylabel('Approval Rate')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Compare model performance with different encoding methods
# Prepare the data
X_num = data[['age', 'income']]
y = data['loan_approved']

# Split the data
X_train_num, X_test_num, y_train, y_test = train_test_split(
    X_num, y, test_size=0.2, random_state=42
)

# Function to evaluate model with different encodings
def evaluate_encodings(cat_cols):
    results = []
    
    # 1. One-Hot Encoding
    X_train_onehot = pd.get_dummies(train_data[cat_cols], drop_first=True)
    X_test_onehot = pd.get_dummies(test_data[cat_cols], drop_first=True)
    
    # Ensure train and test have the same columns
    for col in X_train_onehot.columns:
        if col not in X_test_onehot.columns:
            X_test_onehot[col] = 0
    X_test_onehot = X_test_onehot[X_train_onehot.columns]
    
    # Combine with numerical features
    X_train_combined = pd.concat([X_train_num, X_train_onehot], axis=1)
    X_test_combined = pd.concat([X_test_num, X_test_onehot], axis=1)
    
    # Train model
    model = LogisticRegression(random_state=42)
    model.fit(X_train_combined, y_train)
    accuracy = model.score(X_test_combined, y_test)
    results.append({
        'Encoding': 'One-Hot',
        'Accuracy': accuracy
    })
    
    # 2. Label Encoding
    X_train_label = train_data[cat_cols].copy()
    X_test_label = test_data[cat_cols].copy()
    
    for col in cat_cols:
        le = LabelEncoder()
        X_train_label[col] = le.fit_transform(X_train_label[col])
        X_test_label[col] = le.transform(X_test_label[col])
    
    # Combine with numerical features
    X_train_combined = pd.concat([X_train_num, X_train_label], axis=1)
    X_test_combined = pd.concat([X_test_num, X_test_label], axis=1)
    
    # Train model
    model = LogisticRegression(random_state=42)
    model.fit(X_train_combined, y_train)
    accuracy = model.score(X_test_combined, y_test)
    results.append({
        'Encoding': 'Label',
        'Accuracy': accuracy
    })
    
    # 3. Target Encoding
    X_train_target = train_encoded[[f"{col}_target_encoded" for col in cat_cols]]
    X_test_target = test_encoded[[f"{col}_target_encoded" for col in cat_cols]]
    
    # Combine with numerical features
    X_train_combined = pd.concat([X_train_num, X_train_target], axis=1)
    X_test_combined = pd.concat([X_test_num, X_test_target], axis=1)
    
    # Train model
    model = LogisticRegression(random_state=42)
    model.fit(X_train_combined, y_train)
    accuracy = model.score(X_test_combined, y_test)
    results.append({
        'Encoding': 'Target',
        'Accuracy': accuracy
    })
    
    return pd.DataFrame(results)

# Evaluate the encoding methods
encoding_results = evaluate_encodings(['education', 'job_sector', 'marital_status'])
print("\nEncoding Method Performance Comparison:")
print(encoding_results)

# Visualize the results
plt.figure(figsize=(8, 5))
sns.barplot(x='Encoding', y='Accuracy', data=encoding_results)
plt.title('Model Accuracy with Different Encoding Methods')
plt.ylim(0.5, 1.0)
plt.show()

# Generate a target variable (e.g., loan approval: 0 or 1)
# Let's make it dependent on the features
def generate_target(row):
    # Higher probability of approval for higher education, higher income, etc.
    edu_map = {'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3}
    job_map = {'Other': 0, 'Education': 1, 'Healthcare': 1, 'Finance': 2, 'Tech': 2}
    mari_map = {'Widowed': 0, 'Divorced': 1, 'Single': 2, 'Married': 3}
    
    # Calculate a score based on features
    score = (
        0.3 * (row['income'] / 10000) + 
        0.2 * edu_map[row['education']] + 
        0.2 * job_map[row['job_sector']] + 
        0.1 * mari_map[row['marital_status']] +
        0.2 * (row['age'] / 10)
    )
    
    # Add some randomness
    score += np.random.normal(0, 1)
    
    # Convert to binary outcome
    return 1 if score > 7 else 0

## 4.3.3 Feature Engineering Basics

### What is Feature Engineering?
Feature engineering is the process of using domain knowledge or insights from exploratory data analysis to create *new* features from existing ones, potentially improving model performance. It's often considered more art than science and can be highly impactful.

### Why Do Feature Engineering?
Raw data might not always contain features in the most useful format for a model. Engineered features can:
- Capture complex relationships that the model might not learn on its own
- Highlight important patterns or interactions
- Make the model simpler and more interpretable
- Improve model performance, sometimes dramatically

### Common Feature Engineering Techniques

#### 1. Creating Interaction Features
Combining two features (e.g., multiplying `FeatureA * FeatureB`) to capture their joint effect.

#### 2. Polynomial Features
Creating polynomial terms (e.g., x², x³) from a numerical feature to help linear models capture non-linear relationships.

#### 3. Feature Ratios
Creating meaningful ratios between features (e.g., `Income / Debt` to create a debt-to-income ratio).

#### 4. Binning/Discretization
Converting a continuous numerical feature into discrete categories or bins.

#### 5. Aggregations
For grouped data, calculating statistical measures (mean, sum, count, etc.) within groups.

#### 6. Date/Time Extraction
Extracting components from date/time features (day of week, month, hour, etc.)

#### 7. Text Features
Creating features from text data through techniques like bag-of-words, TF-IDF, or word embeddings.

Let's demonstrate some of these techniques:

In [ ]:
# Load the California Housing dataset again
data = fetch_california_housing()
X = data.data
y = data.target
feature_names = data.feature_names

# Create a DataFrame
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

# Display basic information
print("California Housing Dataset:")
print(df.head())
print("\nBasic Statistics:")
print(df.describe().round(2))

# 1. Creating Interaction Features
# Interaction between bedroom ratio and household size
df['bedrooms_per_person'] = df['AveBedrms'] / df['AveOccup']

# Interaction between income and house age
df['income_age_product'] = df['MedInc'] * df['HouseAge']

# 2. Polynomial Features
# Create polynomial features for income (degree 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
income_poly = poly.fit_transform(df[['MedInc']])

# Add to DataFrame (just the squared term)
df['income_squared'] = income_poly[:, 1]

# 3. Feature Ratios
# Ratio of rooms to bedrooms
df['rooms_per_bedroom'] = df['AveRooms'] / df['AveBedrms']

# 4. Binning/Discretization
# Bin income into categories
df['income_category'] = pd.qcut(df['MedInc'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])

# 5. Custom domain-specific features
# Distance to coast (simplified approximation using just longitude)
df['approx_coast_distance'] = np.abs(df['Longitude'] - (-124))  # Approximate CA coast longitude

# Display the new features
print("\nDataFrame with engineered features:")
print(df.head())

# Visualize some of the engineered features
plt.figure(figsize=(15, 10))

# Original feature - MedInc
plt.subplot(2, 3, 1)
plt.scatter(df['MedInc'], df['target'], alpha=0.5, s=5)
plt.title('Original: MedInc vs Target')
plt.xlabel('Median Income')
plt.ylabel('House Value')

# Squared income
plt.subplot(2, 3, 2)
plt.scatter(df['income_squared'], df['target'], alpha=0.5, s=5)
plt.title('Engineered: Income² vs Target')
plt.xlabel('Income Squared')
plt.ylabel('House Value')

# Income-Age interaction
plt.subplot(2, 3, 3)
plt.scatter(df['income_age_product'], df['target'], alpha=0.5, s=5)
plt.title('Engineered: Income*Age vs Target')
plt.xlabel('Income * Age')
plt.ylabel('House Value')

# Rooms per bedroom
plt.subplot(2, 3, 4)
plt.scatter(df['rooms_per_bedroom'], df['target'], alpha=0.5, s=5)
plt.title('Engineered: Rooms per Bedroom vs Target')
plt.xlabel('Rooms per Bedroom')
plt.ylabel('House Value')

# Bedrooms per person
plt.subplot(2, 3, 5)
plt.scatter(df['bedrooms_per_person'], df['target'], alpha=0.5, s=5)
plt.title('Engineered: Bedrooms per Person vs Target')
plt.xlabel('Bedrooms per Person')
plt.ylabel('House Value')

# Distance to coast
plt.subplot(2, 3, 6)
plt.scatter(df['approx_coast_distance'], df['target'], alpha=0.5, s=5)
plt.title('Engineered: Approx. Distance to Coast vs Target')
plt.xlabel('Approx. Distance to Coast')
plt.ylabel('House Value')

plt.tight_layout()
plt.show()

# Evaluate the impact of feature engineering on model performance
def evaluate_feature_engineering(df):
    # Define feature sets
    original_features = feature_names
    engineered_features = original_features + [
        'bedrooms_per_person', 'income_age_product', 'income_squared',
        'rooms_per_bedroom', 'approx_coast_distance'
    ]
    
    # Prepare data
    X_orig = df[original_features]
    X_eng = df[engineered_features]
    y = df['target']
    
    # Split data
    X_orig_train, X_orig_test, X_eng_train, X_eng_test, y_train, y_test = train_test_split(
        X_orig, X_eng, y, test_size=0.2, random_state=42
    )
    
    # Scale the data
    scaler = StandardScaler()
    X_orig_train_scaled = scaler.fit_transform(X_orig_train)
    X_orig_test_scaled = scaler.transform(X_orig_test)
    
    scaler_eng = StandardScaler()
    X_eng_train_scaled = scaler_eng.fit_transform(X_eng_train)
    X_eng_test_scaled = scaler_eng.transform(X_eng_test)
    
    # Train and evaluate a linear regression model with original features
    model_orig = LinearRegression()
    model_orig.fit(X_orig_train_scaled, y_train)
    y_pred_orig = model_orig.predict(X_orig_test_scaled)
    r2_orig = r2_score(y_test, y_pred_orig)
    rmse_orig = np.sqrt(mean_squared_error(y_test, y_pred_orig))
    
    # Train and evaluate a linear regression model with engineered features
    model_eng = LinearRegression()
    model_eng.fit(X_eng_train_scaled, y_train)
    y_pred_eng = model_eng.predict(X_eng_test_scaled)
    r2_eng = r2_score(y_test, y_pred_eng)
    rmse_eng = np.sqrt(mean_squared_error(y_test, y_pred_eng))
    
    # Return the results
    return {
        'original_r2': r2_orig,
        'engineered_r2': r2_eng,
        'original_rmse': rmse_orig,
        'engineered_rmse': rmse_eng
    }

# Evaluate the impact of feature engineering
results = evaluate_feature_engineering(df)

# Display the results
print("\nImpact of Feature Engineering on Model Performance:")
print(f"Original Features - R²: {results['original_r2']:.4f}, RMSE: {results['original_rmse']:.4f}")
print(f"Engineered Features - R²: {results['engineered_r2']:.4f}, RMSE: {results['engineered_rmse']:.4f}")
print(f"Improvement - R²: {(results['engineered_r2'] - results['original_r2']):.4f}, RMSE: {(results['original_rmse'] - results['engineered_rmse']):.4f}")

# Visualize the results
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.bar(['Original Features', 'Engineered Features'], 
        [results['original_r2'], results['engineered_r2']])
plt.title('R² Score Comparison')
plt.ylabel('R² Score')
plt.ylim(0, 1)

plt.subplot(1, 2, 2)
plt.bar(['Original Features', 'Engineered Features'], 
        [results['original_rmse'], results['engineered_rmse']])
plt.title('RMSE Comparison')
plt.ylabel('RMSE')

plt.tight_layout()
plt.show()

### Key Takeaways on Feature Engineering

1. **Start simple**: Begin with obvious transformations and ratios based on domain knowledge
   
2. **Visualize features**: Plot engineered features against the target to check if they have a clear relationship
   
3. **Use domain knowledge**: Understanding the problem domain can suggest useful feature transformations
   
4. **Evaluate impact**: Always measure the impact of engineered features on model performance
   
5. **Beware of overfitting**: Complex feature engineering can lead to overfitting; validate on a separate test set
   
6. **Document your steps**: Keep track of which engineered features were effective

## Knowledge Check

1. Why is feature scaling necessary for algorithms like K-Nearest Neighbors but not for decision trees?
   
2. What's the key difference between StandardScaler and MinMaxScaler, and when would you choose one over the other?
   
3. Why might one-hot encoding be problematic for categorical features with high cardinality (many unique values)?
   
4. In what situation would label encoding be appropriate for a categorical feature?
   
5. How does feature engineering differ from other preprocessing steps like scaling and encoding?

## Mini-Challenges

### Challenge 1: Feature Scaling Comparison
Compare the impact of different scaling methods (StandardScaler, MinMaxScaler, RobustScaler) on a KNN classifier's performance using the Breast Cancer dataset.

In [ ]:
# Starter code
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import accuracy_score

# Load data
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Your code here to compare scalers with KNN

In [ ]:
### Challenge 2: Categorical Encoding Experiment
Create a synthetic dataset with a mix of numerical and categorical features. Experiment with different encoding methods and analyze their impact on model performance.

In [ ]:
# Starter code
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Your code here to create a dataset and compare encodings

### Challenge 3: Feature Engineering for Time Series
Using a synthetic time series dataset, create engineered features like:
- Day of week
- Month of year
- Is weekend/holiday
- Rolling averages
Then build a regression model and evaluate the impact of these features.

In [ ]:
# Starter code
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Create a synthetic time series dataset
dates = pd.date_range(start='2020-01-01', end='2022-12-31', freq='D')
np.random.seed(42)
values = np.random.normal(loc=100, scale=20, size=len(dates))

# Synthetic seasonality and trend
trend = np.linspace(0, 30, len(dates))  # Increasing trend
day_of_year = np.array([d.dayofyear for d in dates])
seasonality = 15 * np.sin(2 * np.pi * day_of_year / 365)  # Yearly seasonality
weekend_effect = np.array([5 if d.dayofweek >= 5 else 0 for d in dates])  # Weekend effect

# Combine components
values = values + trend + seasonality + weekend_effect

# Create DataFrame
df = pd.DataFrame({'date': dates, 'value': values})

# Your code here to engineer time-based features and build a model

## Practical Connections

- **Finance**: In credit scoring models, financial ratios like debt-to-income are often more predictive than raw debt and income values separately.

- **E-commerce**: Feature interaction between user browsing history and product categories can improve recommendation systems significantly.

- **Healthcare**: Combining patient vital signs into medically meaningful ratios (e.g., BMI from height and weight) can create more informative predictors for medical diagnoses.

- **Real Estate**: In housing price prediction, features like price-per-square-foot or rooms-per-person often show stronger relationships with target values than the individual components.

- **Manufacturing**: In quality control, polynomial features of manufacturing parameters can help identify non-linear relationships with defect rates.

- **Marketing**: For customer segmentation, binning continuous features like age or spending into meaningful categories can make models more interpretable and actionable.

- **Transportation**: In traffic prediction, extracting time-based features like hour-of-day and day-of-week from timestamps can capture recurring traffic patterns.

In [ ]:
# Compare the distributions before and after scaling
plt.figure(figsize=(15, 10))
for i, feature in enumerate(selected_features):
    plt.subplot(2, 2, i+1)
    
    # Original data
    sns.histplot(df[feature], kde=True, alpha=0.5, color='blue', label='Original')
    
    # Standardized data
    sns.histplot(df_std[feature], kde=True, alpha=0.5, color='red', label='Standardized')
    
    # Min-Max scaled data
    sns.histplot(df_minmax[feature], kde=True, alpha=0.5, color='green', label='Min-Max')
    
    plt.title(f'Distribution of {feature}')
    plt.legend()

plt.tight_layout()
plt.show()

# Compare statistics after scaling
print("\nStandardized Data Statistics:")
print(df_std.describe().round(2))

print("\nMin-Max Scaled Data Statistics:")
print(df_minmax.describe().round(2))

In [ ]:
# Create a sample dataset with categorical features
np.random.seed(42)
n_samples = 500

# Create a DataFrame with a mix of numerical and categorical features
data = pd.DataFrame({
    'age': np.random.normal(40, 10, n_samples),
    'income': np.random.normal(60000, 15000, n_samples),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples),
    'job_sector': np.random.choice(['Tech', 'Finance', 'Healthcare', 'Education', 'Other'], n_samples),
    'marital_status': np.random.choice(['Single', 'Married', 'Divorced', 'Widowed'], n_samples)
})

# Add a binary target variable (loan approval)
data['loan_approved'] = data.apply(generate_target, axis=1)

# Display the first few rows
print("Sample data:")
print(data.head())

# Check data types and categorical feature counts
print("\nData types:")
print(data.dtypes)

print("\nCategorical feature value counts:")
for col in ['education', 'job_sector', 'marital_status']:
    print(f"\n{col} value counts:")
    print(data[col].value_counts())

# 1. One-Hot Encoding using pandas get_dummies
data_onehot = pd.get_dummies(data, columns=['education', 'job_sector', 'marital_status'], drop_first=False)

# Display the first few rows after one-hot encoding
print("\nAfter One-Hot Encoding (first 5 columns):")
print(data_onehot.iloc[:5, :10])
print(f"Shape after one-hot encoding: {data_onehot.shape}")

# 2. Label Encoding
from sklearn.preprocessing import LabelEncoder

data_label = data.copy()
label_encoders = {}

for col in ['education', 'job_sector', 'marital_status']:
    label_encoders[col] = LabelEncoder()
    data_label[col] = label_encoders[col].fit_transform(data_label[col])

# Display the first few rows after label encoding
print("\nAfter Label Encoding:")
print(data_label.head())

# Show mapping for education
print("\nLabel Encoder Mapping for 'education':")
for i, label in enumerate(label_encoders['education'].classes_):
    print(f"{label} -> {i}")

# 3. Target Encoding (mean encoding)
def target_encode(train_df, test_df, cols, target_col, min_samples=10, smoothing=10):
    # Dictionary to store the encoders
    encoders = {}
    
    # Process each column
    for col in cols:
        # Calculate the global mean
        global_mean = train_df[target_col].mean()
        
        # Group by the column and calculate the mean of the target
        category_means = train_df.groupby(col)[target_col].agg(['mean', 'count'])
        
        # Apply smoothing
        smoothed_means = (category_means['mean'] * category_means['count'] + 
                         global_mean * smoothing) / (category_means['count'] + smoothing)
        
        # Create the encoder dictionary
        encoders[col] = smoothed_means.to_dict()
        
        # Apply encoding to train data
        train_df[f"{col}_target_encoded"] = train_df[col].map(encoders[col]).fillna(global_mean)
        
        # Apply encoding to test data
        test_df[f"{col}_target_encoded"] = test_df[col].map(encoders[col]).fillna(global_mean)
    
    return train_df, test_df, encoders

# Split the data for target encoding demonstration
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Apply target encoding
train_encoded, test_encoded, encoders = target_encode(
    train_data.copy(), 
    test_data.copy(), 
    cols=['education', 'job_sector', 'marital_status'],
    target_col='loan_approved'
)

# Display the first few rows after target encoding
print("\nAfter Target Encoding (Training Data):")
print(train_encoded.head())

# Compare the different encoding methods
plt.figure(figsize=(15, 5))

# Original distribution (for education)
plt.subplot(1, 3, 1)
sns.countplot(x='education', data=data)
plt.title('Original Categories')
plt.xticks(rotation=45)

# Label Encoded
plt.subplot(1, 3, 2)
sns.countplot(x='education', data=data_label)
plt.title('Label Encoded')

# Target Encoded
plt.subplot(1, 3, 3)
sns.boxplot(x='education', y='education_target_encoded', data=train_encoded)
plt.title('Target Encoded')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Visualize the relationship between categorical features and target
plt.figure(figsize=(15, 5))

# Education vs. Target
plt.subplot(1, 3, 1)
target_by_edu = data.groupby('education')['loan_approved'].mean().reset_index()
sns.barplot(x='education', y='loan_approved', data=target_by_edu)
plt.title('Loan Approval Rate by Education')
plt.ylabel('Approval Rate')
plt.xticks(rotation=45)

# Job Sector vs. Target
plt.subplot(1, 3, 2)
target_by_job = data.groupby('job_sector')['loan_approved'].mean().reset_index()
sns.barplot(x='job_sector', y='loan_approved', data=target_by_job)
plt.title('Loan Approval Rate by Job Sector')
plt.ylabel('Approval Rate')
plt.xticks(rotation=45)

# Marital Status vs. Target
plt.subplot(1, 3, 3)
target_by_marital = data.groupby('marital_status')['loan_approved'].mean().reset_index()
sns.barplot(x='marital_status', y='loan_approved', data=target_by_marital)
plt.title('Loan Approval Rate by Marital Status')
plt.ylabel('Approval Rate')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Compare model performance with different encoding methods
# Prepare the data
X_num = data[['age', 'income']]
y = data['loan_approved']

# Split the data
X_train_num, X_test_num, y_train, y_test = train_test_split(
    X_num, y, test_size=0.2, random_state=42
)

# Function to evaluate model with different encodings
def evaluate_encodings(cat_cols):
    results = []
    
    # 1. One-Hot Encoding
    X_train_onehot = pd.get_dummies(train_data[cat_cols], drop_first=True)
    X_test_onehot = pd.get_dummies(test_data[cat_cols], drop_first=True)
    
    # Ensure train and test have the same columns
    for col in X_train_onehot.columns:
        if col not in X_test_onehot.columns:
            X_test_onehot[col] = 0
    X_test_onehot = X_test_onehot[X_train_onehot.columns]
    
    # Combine with numerical features
    X_train_combined = pd.concat([X_train_num, X_train_onehot], axis=1)
    X_test_combined = pd.concat([X_test_num, X_test_onehot], axis=1)
    
    # Train model
    model = LogisticRegression(random_state=42)
    model.fit(X_train_combined, y_train)
    accuracy = model.score(X_test_combined, y_test)
    results.append({
        'Encoding': 'One-Hot',
        'Accuracy': accuracy
    })
    
    # 2. Label Encoding
    X_train_label = train_data[cat_cols].copy()
    X_test_label = test_data[cat_cols].copy()
    
    for col in cat_cols:
        le = LabelEncoder()
        X_train_label[col] = le.fit_transform(X_train_label[col])
        X_test_label[col] = le.transform(X_test_label[col])
    
    # Combine with numerical features
    X_train_combined = pd.concat([X_train_num, X_train_label], axis=1)
    X_test_combined = pd.concat([X_test_num, X_test_label], axis=1)
    
    # Train model
    model = LogisticRegression(random_state=42)
    model.fit(X_train_combined, y_train)
    accuracy = model.score(X_test_combined, y_test)
    results.append({
        'Encoding': 'Label',
        'Accuracy': accuracy
    })
    
    # 3. Target Encoding
    X_train_target = train_encoded[[f"{col}_target_encoded" for col in cat_cols]]
    X_test_target = test_encoded[[f"{col}_target_encoded" for col in cat_cols]]
    
    # Combine with numerical features
    X_train_combined = pd.concat([X_train_num, X_train_target], axis=1)
    X_test_combined = pd.concat([X_test_num, X_test_target], axis=1)
    
    # Train model
    model = LogisticRegression(random_state=42)
    model.fit(X_train_combined, y_train)
    accuracy = model.score(X_test_combined, y_test)
    results.append({
        'Encoding': 'Target',
        'Accuracy': accuracy
    })
    
    return pd.DataFrame(results)

# Evaluate the encoding methods
encoding_results = evaluate_encodings(['education', 'job_sector', 'marital_status'])
print("\nEncoding Method Performance Comparison:")
print(encoding_results)

# Visualize the results
plt.figure(figsize=(8, 5))
sns.barplot(x='Encoding', y='Accuracy', data=encoding_results)
plt.title('Model Accuracy with Different Encoding Methods')
plt.ylim(0.5, 1.0)
plt.show()

# Generate a target variable (e.g., loan approval: 0 or 1)
# Let's make it dependent on the features
def generate_target(row):
    # Higher probability of approval for higher education, higher income, etc.
    edu_map = {'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3}
    job_map = {'Other': 0, 'Education': 1, 'Healthcare': 1, 'Finance': 2, 'Tech': 2}
    mari_map = {'Widowed': 0, 'Divorced': 1, 'Single': 2, 'Married': 3}
    
    # Calculate a score based on features
    score = (
        0.3 * (row['income'] / 10000) + 
        0.2 * edu_map[row['education']] + 
        0.2 * job_map[row['job_sector']] + 
        0.1 * mari_map[row['marital_status']] +
        0.2 * (row['age'] / 10)
    )
    
    # Add some randomness
    score += np.random.normal(0, 1)
    
    # Convert to binary outcome
    return 1 if score > 7 else 0